Daisyworld: визуализация состояний модели для разных параметров

В этом эксперименте выполняется серия симуляций модели Daisyworld
для различных комбинаций параметров. Для каждой симуляции
сохраняются визуализации состояния системы на разных шагах
эволюции модели.

## Подключение пакетов

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))

using DrWatson
@quickactivate "project"

using Agents
using DataFrames
using Plots
using CairoMakie

  Activating project at `~/work/study/2026--1/2026-1-study-simulation-modeling/labs/lab03/project`
[ Info: Precompiling Agents [46ada45e-f475-11e8-01d0-f70cc89e6671] (cache misses: wrong dep version loaded (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling AgentsVisualizations [eaec18ce-d6c3-5e41-ba47-385c9e61b392] (cache misses: wrong dep version loaded (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


## Подключение реализации модели

In [2]:
include(srcdir("daisyworld.jl"))

  Activating project at `~/work/study/2026--1/2026-1-study-simulation-modeling/labs/lab03/project`


daisyworld (generic function with 1 method)

## Параметры эксперимента

Некоторые параметры заданы как массивы значений.
Это означает, что будут автоматически сформированы
все возможные комбинации этих параметров.

In [3]:
param_dict = Dict(
    :griddims => (30, 30),
    :max_age => [25, 40],
    :init_white => [0.2, 0.8],
    :init_black => 0.2,
    :albedo_white => 0.75,
    :albedo_black => 0.25,
    :surface_albedo => 0.4,
    :solar_change => 0.005,
    :solar_luminosity => 1.0,
    :scenario => :default,
    :seed => 165,
)

Dict{Symbol, Any} with 11 entries:
  :solar_luminosity => 1.0
  :scenario         => :default
  :griddims         => (30, 30)
  :max_age          => [25, 40]
  :surface_albedo   => 0.4
  :init_white       => [0.2, 0.8]
  :albedo_white     => 0.75
  :init_black       => 0.2
  :solar_change     => 0.005
  :albedo_black     => 0.25
  :seed             => 165

## Формирование списка комбинаций параметров

In [4]:
params_list = dict_list(param_dict)

4-element Vector{Dict{Symbol, Any}}:
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 25, :surface_albedo => 0.4, :init_white => 0.2, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 25, :surface_albedo => 0.4, :init_white => 0.8, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 40, :surface_albedo => 0.4, :init_white => 0.2, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)
 Dict(:solar_luminosity => 1.0, :scenario => :default, :griddims => (30, 30), :max_age => 40, :surface_albedo => 0.4, :init_white => 0.8, :albedo_white => 0.75, :init_black => 0.2, :solar_change => 0.005, :albedo_black => 0.25…)

## Проведение серии экспериментов

In [11]:
for params in params_list

    model = daisyworld(; params...)

    ## Настройка визуализации агентов
    daisycolor(a::Daisy) = a.breed

    plotkwargs = (
        agent_color = daisycolor,
        agent_size = 20,
        agent_marker = '✿',
        heatarray = :temperature,
        heatkwargs = (colorrange = (-20, 60),),
    )
    ## Начальное состояние системы
    plt1, _ = abmplot(model; plotkwargs...)
    ## Состояние после 5 шагов симуляции
    step!(model, 5)
    
    plt2, _ = abmplot(
        model;
        heatarray = model.temperature,
        plotkwargs...
    )
    ## Состояние после 40 шагов симуляции
    step!(model, 40)

    plt3, _ = abmplot(
        model;
        heatarray = model.temperature,
        plotkwargs...
    )
    ## Сохранение изображений
    plt1_name = savename("daisyworld", params) * "_step01.png"
    plt2_name = savename("daisyworld", params) * "_step05.png"
    plt3_name = savename("daisyworld", params) * "_step40.png"

    save(plotsdir(plt1_name), plt1)
    save(plotsdir(plt2_name), plt2)
    save(plotsdir(plt3_name), plt3)
end

## Анализ результатов

Для каждой комбинации параметров сохраняются три изображения:

1. начальное состояние модели
2. состояние после нескольких шагов симуляции
3. состояние после более длительной эволюции

На визуализациях отображаются:
- расположение ромашек
- их тип (цвет)
- распределение температуры поверхности

Сравнение полученных изображений позволяет проследить,
как система развивается во времени и как параметры модели
влияют на пространственное распределение ромашек
и температурное поле.